In [44]:
import os
import pandas as pd
from sklearn.utils import resample

In [45]:
# Define the folder paths
base_path = r"./raw/watch"
gyro_folder = os.path.join(base_path, "gyro")
accel_folder = os.path.join(base_path, "accel")

In [46]:
# Function to read and concatenate all files in a folder
def read_sensor_data(folder_path, columns):
    all_files = [os.path.join(folder_path, f) for f in os.listdir(folder_path) if f.endswith(".txt")]
    dataframes = []

    for file in all_files:
        df = pd.read_csv(file, header=None, names=columns)
        dataframes.append(df)

    return pd.concat(dataframes, ignore_index=True)

In [47]:
# Read data from both folders
gyro_columns = ["Subject-id", "Activity Label", "Timestamp", "gyro_x", "gyro_y", "gyro_z"]
accel_columns = ["Subject-id", "Activity Label", "Timestamp", "acc_x", "acc_y", "acc_z"]

In [48]:
gyro_data = read_sensor_data(gyro_folder, gyro_columns)
accel_data = read_sensor_data(accel_folder, accel_columns)

In [49]:
# Merge the two DataFrames on the Timestamp column
merged_data = pd.merge(gyro_data, accel_data, on="Timestamp", suffixes=("_gyro", "_accel"))

In [50]:
# Display the first few rows of the merged DataFrame
merged_data.head()

,Subject-id_gyro,Activity Label_gyro,Timestamp,gyro_x,gyro_y,gyro_z,Subject-id_accel,Activity Label_accel,acc_x,acc_y,acc_z
0,1650,A,2428057182945480,0.334463,0.102968,-0.3460815;,1650,A,-3.302951,-6.823175,-4.011485;
1,1650,A,2428057232445480,0.086256,0.719756,0.45393208;,1650,A,-2.876783,-8.365042,-7.171831;
2,1650,A,2428057281945480,-0.040510,1.101121,1.0462191;,1650,A,-2.254291,-8.566154,-8.153454;
3,1650,A,2428057331445480,-0.224801,1.382351,0.94288844;,1650,A,-3.264644,-10.953174,-8.826224;
4,1650,A,2428057380945480,-0.347306,0.898720,0.62863547;,1650,A,-2.261473,-8.683471,-9.175778;


In [51]:
merged_data.drop(columns=["Subject-id_accel", "Activity Label_accel"], inplace=True)
merged_data.head()

,Subject-id_gyro,Activity Label_gyro,Timestamp,gyro_x,gyro_y,gyro_z,acc_x,acc_y,acc_z
0,1650,A,2428057182945480,0.334463,0.102968,-0.3460815;,-3.302951,-6.823175,-4.011485;
1,1650,A,2428057232445480,0.086256,0.719756,0.45393208;,-2.876783,-8.365042,-7.171831;
2,1650,A,2428057281945480,-0.040510,1.101121,1.0462191;,-2.254291,-8.566154,-8.153454;
3,1650,A,2428057331445480,-0.224801,1.382351,0.94288844;,-3.264644,-10.953174,-8.826224;
4,1650,A,2428057380945480,-0.347306,0.898720,0.62863547;,-2.261473,-8.683471,-9.175778;


In [52]:
merged_data.columns = ["user_id", "activity", "timestamp", "gyro_x", "gyro_y", "gyro_z", "acc_x", "acc_y", "acc_z"]

In [53]:
merged_data.head()

,user_id,activity,timestamp,gyro_x,gyro_y,gyro_z,acc_x,acc_y,acc_z
0,1650,A,2428057182945480,0.334463,0.102968,-0.3460815;,-3.302951,-6.823175,-4.011485;
1,1650,A,2428057232445480,0.086256,0.719756,0.45393208;,-2.876783,-8.365042,-7.171831;
2,1650,A,2428057281945480,-0.040510,1.101121,1.0462191;,-2.254291,-8.566154,-8.153454;
3,1650,A,2428057331445480,-0.224801,1.382351,0.94288844;,-3.264644,-10.953174,-8.826224;
4,1650,A,2428057380945480,-0.347306,0.898720,0.62863547;,-2.261473,-8.683471,-9.175778;


In [54]:
# Map activity labels to new values
activity_map = {
    "A": "Walking",
    "B": "Jogging",
    "C": "Stairs",
    "D": "Sitting",
    "E": "Standing"
}

merged_data["activity"] = merged_data["activity"].map(activity_map)

merged_data['gyro_z'] = merged_data['gyro_z'].astype(str).str.replace(';', '', regex=False)
merged_data['acc_z'] = merged_data['acc_z'].astype(str).str.replace(';', '', regex=False)

In [55]:
merged_data

,user_id,activity,timestamp,gyro_x,gyro_y,gyro_z,acc_x,acc_y,acc_z
0,1650,Walking,2428057182945480,0.334463,0.102968,-0.3460815,-3.302951,-6.823175,-4.011485
1,1650,Walking,2428057232445480,0.086256,0.719756,0.45393208,-2.876783,-8.365042,-7.171831
2,1650,Walking,2428057281945480,-0.040510,1.101121,1.0462191,-2.254291,-8.566154,-8.153454
3,1650,Walking,2428057331445480,-0.224801,1.382351,0.94288844,-3.264644,-10.953174,-8.826224
4,1650,Walking,2428057380945480,-0.347306,0.898720,0.62863547,-2.261473,-8.683471,-9.175778
...,...,...,...,...,...,...,...,...,...
3370856,1636,NaN,508781301831491,-0.616779,2.701672,-2.107579,0.395642,0.167744,5.5663695
3370857,1636,NaN,508781351754011,0.672191,3.126713,-2.3014572,-8.364742,4.585046,7.673267
3370858,1636,NaN,508781401676531,1.521207,4.137649,-1.9222231,-9.341577,2.475754,8.343643
3370859,1636,NaN,508781451599051,2.963575,1.161300,0.51829773,-9.492412,5.439776,6.2439284


In [56]:
# Drop rows with Activity Label not in the map
merged_data = merged_data.dropna()
merged_data

,user_id,activity,timestamp,gyro_x,gyro_y,gyro_z,acc_x,acc_y,acc_z
0,1650,Walking,2428057182945480,0.334463,0.102968,-0.3460815,-3.302951,-6.823175,-4.011485
1,1650,Walking,2428057232445480,0.086256,0.719756,0.45393208,-2.876783,-8.365042,-7.171831
2,1650,Walking,2428057281945480,-0.040510,1.101121,1.0462191,-2.254291,-8.566154,-8.153454
3,1650,Walking,2428057331445480,-0.224801,1.382351,0.94288844,-3.264644,-10.953174,-8.826224
4,1650,Walking,2428057380945480,-0.347306,0.898720,0.62863547,-2.261473,-8.683471,-9.175778
...,...,...,...,...,...,...,...,...,...
3324014,1636,Standing,507714885935888,-0.098618,-0.042543,0.36633858,-9.128492,4.699967,-2.011279
3324015,1636,Standing,507714935854278,-1.187318,0.293015,0.95010346,-9.061455,11.506683,1.2831424
3324016,1636,Standing,507714985772668,-1.601706,2.294647,1.6808748,-8.685565,12.418874,-0.72080433
3324017,1636,Standing,507715035691058,-3.049400,2.525809,2.3168378,-7.445369,10.465205,-2.6720788


In [ ]:
# Function to balance the dataset
def balance_dataset(df, class_column):
    # Count the occurrences of each class
    class_counts = df[class_column].value_counts()
    min_count = class_counts.min()

    print(min_count)

    # Resample each class to have the same number of occurrences as the class with the smallest count
    balanced_df = pd.DataFrame()
    for activity in class_counts.index:
        activity_df = df[df[class_column] == activity]
        resampled_df = resample(activity_df, replace=False, n_samples=min_count, random_state=42)
        balanced_df = pd.concat([balanced_df, resampled_df], ignore_index=True)
    
    return balanced_df

# Balance the dataset
balanced_df = balance_dataset(merged_data, "activity")

181618


In [58]:
balanced_df

,user_id,activity,timestamp,gyro_x,gyro_y,gyro_z,acc_x,acc_y,acc_z
0,1629,Standing,378074063838339,-0.041636,-0.006295,-0.0073846793,9.086594,-2.904466,-2.4103625
1,1608,Standing,718310555580760,0.365014,-0.144451,0.041347172,9.242966,-2.845060,-0.17597382
2,1603,Standing,7416336313569,-0.025930,-0.005946,-0.024872327,9.492560,-2.577508,-0.24420857
3,1649,Standing,2720543059666407,-0.003721,-0.024820,-0.015590678,7.316231,-6.360945,0.720954
4,1629,Standing,378117495393979,-0.028996,0.023560,0.0031814121,9.326014,-2.851793,-0.9115923
...,...,...,...,...,...,...,...,...,...
908085,1606,Jogging,802543879346752,-5.746271,2.084026,-0.110875115,-1.180192,-7.747337,5.6450787
908086,1624,Jogging,292875439441796,0.669892,-1.324213,0.36815777,6.526893,0.179864,2.53531
908087,1618,Jogging,15348150236711,-0.825846,1.557195,-0.21792008,5.694758,-19.367146,7.1431007
908088,1630,Jogging,1334534131800988,0.071009,-5.307314,2.5128126,10.305392,-19.350536,-16.932991


In [59]:
# Save the merged DataFrame to a CSV file (optional)
output_path = os.path.join(base_path, "WISDM_ar_v1.1_balanced")
merged_data.to_csv(output_path, index=False)